###Initialize

In [0]:
#libraries
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType
from pyspark.sql import functions as F

###Reading from bronze level

In [0]:
df = spark.table('etl_project.bronze.cust_info')

###Data Transformation

In [0]:
#trim the DataFrame

for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))
    
df.display()

In [0]:
# rows count
df.count()

In [0]:
df.columns

In [0]:
#remove duplicates from cst_id
df = df.dropDuplicates(['cst_id'])
df.count()

In [0]:


df = df.withColumns(
    {
        "cst_marital_status": F.when(
            F.upper(F.col("cst_marital_status")) == "S", "Single"
        )
        .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
        .otherwise("n/a"),
        "cst_gndr": F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
        .when(F.upper(F.col("cst_gndr")) == "M", "Male")
        .otherwise("n/a"),
    }
)


df.display()

In [0]:
# rename all the columns names
rename_map = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date",
}

df = df.withColumnsRenamed(rename_map)

df.display()

###Write into Silver Table


In [0]:
df.write.mode("overwrite").saveAsTable("etl_project.silver.silver_cust_info")

In [0]:
%sql 
select * from etl_project.silver.silver_cust_info;

###For the second table

###Initialize

In [0]:
#libraries
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType
from pyspark.sql import functions as F

In [0]:
rename_map = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date",
}

###Load data from bronze

In [0]:
df = spark.table('etl_project.bronze.prd_info')
df.display()

###Transformation

In [0]:
#trim the DataFrame

for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))
    
df.display()

In [0]:
df.columns

In [0]:
#Relace the values in the prd_line column for better readability

df = df.withColumns(
    {
        "prd_line": F.when(F.upper(F.col("prd_line")) == "M", "Mountain")
        .when(F.upper(F.col("prd_line")) == "R", "Road")
        .when(F.upper(F.col("prd_line")) == "S", "Sports")
        .when(F.upper(F.col("prd_line")) == "T", "Touring")
        .otherwise("n/a"),
    }
)

df.display()

In [0]:
# Fill nulls with 0 specifically for the 'prd_cost' column
df = df.fillna({"prd_cost": 0})

df.display()

In [0]:
# Apply the column renames defined earlier
df = df.withColumnsRenamed(rename_map)

df.display()

###Write to the silver

In [0]:
df.write.mode("overwrite").saveAsTable("etl_project.silver.silver_prd_info")

In [0]:
%sql
select * from etl_project.silver.silver_prd_info;